In [19]:
import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold


DATASET_NAME = "Whisper Base 0.5s"

DATASET_PATH = Path(
    "/Users/bhavaykhatri/Desktop/whisper_base/"
    "singBAP_dataset_whisper_whisper-base_0.5s.parquet"
)

TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]


def decode_embedding(value):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        value,
        dtype=np.float32,
    ).reshape(-1)


# Load and filter the same data as the baseline
df = pd.read_parquet(DATASET_PATH)

df = df[
    df["experience"].isin(
        ["intermediate", "professional"]
    )
].copy()

df = df[
    df["condition"].isin(TARGET_CLASSES)
].copy()

df = df.reset_index(drop=True)


# Create features, labels, and recording groups
X = np.vstack(
    df["embedding"].apply(
        decode_embedding
    )
)

y = (
    df["condition"]
    .astype(str)
    .to_numpy()
)

groups = (
    df["filename"]
    .astype(str)
    .to_numpy()
)


# Create the same grouped outer split
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    outer_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

groups_train = groups[train_idx]
groups_test = groups[test_idx]


print("Full dataset:", X.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print(
    "Shared outer recordings:",
    len(
        set(groups_train)
        & set(groups_test)
    ),
)

Full dataset: (28418, 512)
Training set: (22735, 512)
Test set: (5683, 512)
Shared outer recordings: 0


In [21]:
groups_train = groups[train_idx]

inner_splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=43,
)

feature_train_relative_idx, validation_relative_idx = next(
    inner_splitter.split(
        X_train,
        y_train,
        groups=groups_train,
    )
)

X_feature_train = X_train[
    feature_train_relative_idx
]

y_feature_train = y_train[
    feature_train_relative_idx
]

X_validation = X_train[
    validation_relative_idx
]

y_validation = y_train[
    validation_relative_idx
]

feature_train_groups = groups_train[
    feature_train_relative_idx
]

validation_groups = groups_train[
    validation_relative_idx
]

shared_inner_recordings = (
    set(feature_train_groups)
    & set(validation_groups)
)

print(
    "Feature-selection training:",
    X_feature_train.shape,
)

print(
    "Validation:",
    X_validation.shape,
)

print(
    "Shared inner recordings:",
    len(shared_inner_recordings),
)

Feature-selection training: (17049, 512)
Validation: (5686, 512)
Shared inner recordings: 0


In [22]:
SEARCH_MODEL = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        dual=False,
        tol=1e-3,
        max_iter=5000,
        random_state=42,
    ),
)


def evaluate_feature_indices(feature_indices):
    feature_indices = np.asarray(
        feature_indices,
        dtype=int,
    )

    if feature_indices.size == 0:
        return np.nan

    model = clone(SEARCH_MODEL)

    model.fit(
        X_feature_train[:, feature_indices],
        y_feature_train,
    )

    validation_predictions = model.predict(
        X_validation[:, feature_indices]
    )

    return f1_score(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )

In [23]:
selection_results = []
selected_feature_sets = {}


def record_selection(
    method,
    family,
    feature_indices,
    validation_macro_f1,
    elapsed_time,
):
    feature_indices = np.asarray(
        feature_indices,
        dtype=int,
    )

    selected_feature_sets[method] = (
        feature_indices
    )

    selection_results.append({
        "Method": method,
        "Family": family,
        "Selected Features": len(
            feature_indices
        ),
        "Validation Macro F1": (
            validation_macro_f1
        ),
        "Selection Time": elapsed_time,
    })

    print(
        f"{method}: "
        f"{len(feature_indices)} features, "
        f"validation Macro F1="
        f"{validation_macro_f1:.4f}, "
        f"time={elapsed_time:.2f}s"
    )

In [24]:
all_feature_indices = np.arange(
    X_feature_train.shape[1]
)

start_time = time.time()

all_features_f1 = evaluate_feature_indices(
    all_feature_indices
)

elapsed_time = time.time() - start_time

record_selection(
    method="All Features",
    family="Baseline",
    feature_indices=all_feature_indices,
    validation_macro_f1=all_features_f1,
    elapsed_time=elapsed_time,
)

All Features: 512 features, validation Macro F1=0.2900, time=45.30s


In [25]:
ANOVA_K_VALUES = [
    64,
    128,
    192,
    256,
    320,
    384,
    448,
]

for k in ANOVA_K_VALUES:
    if k >= X_feature_train.shape[1]:
        continue

    start_time = time.time()

    selector = SelectKBest(
        score_func=f_classif,
        k=k,
    )

    selector.fit(
        X_feature_train,
        y_feature_train,
    )

    selected_indices = (
        selector.get_support(
            indices=True
        )
    )

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    elapsed_time = (
        time.time() - start_time
    )

    record_selection(
        method=f"ANOVA K={k}",
        family="ANOVA",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=elapsed_time,
    )

ANOVA K=64: 64 features, validation Macro F1=0.2099, time=0.66s
ANOVA K=128: 128 features, validation Macro F1=0.2213, time=2.16s
ANOVA K=192: 192 features, validation Macro F1=0.2424, time=5.65s
ANOVA K=256: 256 features, validation Macro F1=0.2531, time=9.96s
ANOVA K=320: 320 features, validation Macro F1=0.2644, time=16.31s
ANOVA K=384: 384 features, validation Macro F1=0.2680, time=22.05s
ANOVA K=448: 448 features, validation Macro F1=0.2796, time=36.36s


In [26]:
L1_C_VALUES = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
]

l1_scaler = StandardScaler()

X_feature_train_l1 = (
    l1_scaler.fit_transform(
        X_feature_train
    )
)

for c_value in L1_C_VALUES:
    start_time = time.time()

    l1_model = LinearSVC(
        C=c_value,
        penalty="l1",
        dual=False,
        class_weight="balanced",
        tol=1e-3,
        max_iter=5000,
        random_state=42,
    )

    l1_model.fit(
        X_feature_train_l1,
        y_feature_train,
    )

    selected_mask = np.any(
        np.abs(l1_model.coef_) > 1e-8,
        axis=0,
    )

    selected_indices = np.flatnonzero(
        selected_mask
    )

    if len(selected_indices) == 0:
        print(
            f"L1 C={c_value} selected "
            "zero features; skipped."
        )
        continue

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    elapsed_time = (
        time.time() - start_time
    )

    record_selection(
        method=f"L1-SVM C={c_value}",
        family="L1-SVM",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=elapsed_time,
    )

L1-SVM C=0.001: 40 features, validation Macro F1=0.1930, time=1.94s
L1-SVM C=0.003: 138 features, validation Macro F1=0.2337, time=7.97s
L1-SVM C=0.01: 330 features, validation Macro F1=0.2629, time=105.95s
L1-SVM C=0.03: 459 features, validation Macro F1=0.2878, time=140.27s


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


L1-SVM C=0.1: 509 features, validation Macro F1=0.2908, time=568.01s


In [27]:
def sequential_forward_selection(
    candidate_indices,
    maximum_selected=15,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    selected = []
    remaining = candidate_indices.copy()

    best_overall_score = -np.inf
    best_overall_subset = None

    history = []

    for step in range(
        min(
            maximum_selected,
            len(candidate_indices),
        )
    ):
        best_step_feature = None
        best_step_score = -np.inf

        for feature_index in remaining:
            trial_subset = (
                selected
                + [feature_index]
            )

            score = evaluate_feature_indices(
                trial_subset
            )

            if score > best_step_score:
                best_step_score = score
                best_step_feature = (
                    feature_index
                )

        selected.append(
            best_step_feature
        )

        remaining.remove(
            best_step_feature
        )

        history.append({
            "Step": step + 1,
            "Added Feature": (
                best_step_feature
            ),
            "Selected Features": len(
                selected
            ),
            "Validation Macro F1": (
                best_step_score
            ),
        })

        print(
            f"Step {step + 1}: "
            f"added feature "
            f"{best_step_feature}, "
            f"Macro F1="
            f"{best_step_score:.4f}"
        )

        if (
            best_step_score
            > best_overall_score
        ):
            best_overall_score = (
                best_step_score
            )

            best_overall_subset = (
                selected.copy()
            )

    return (
        np.asarray(
            best_overall_subset,
            dtype=int,
        ),
        best_overall_score,
        pd.DataFrame(history),
    )

In [28]:
SFS_PREFILTER_K = 30
SFS_MAXIMUM_SELECTED = 15

sfs_prefilter = SelectKBest(
    score_func=f_classif,
    k=SFS_PREFILTER_K,
)

sfs_prefilter.fit(
    X_feature_train,
    y_feature_train,
)

sfs_candidate_indices = (
    sfs_prefilter.get_support(
        indices=True
    )
)

start_time = time.time()

(
    sequential_indices,
    sequential_validation_f1,
    sequential_history_df,
) = sequential_forward_selection(
    candidate_indices=(
        sfs_candidate_indices
    ),
    maximum_selected=(
        SFS_MAXIMUM_SELECTED
    ),
)

elapsed_time = time.time() - start_time

record_selection(
    method="Sequential Forward",
    family="Sequential",
    feature_indices=sequential_indices,
    validation_macro_f1=(
        sequential_validation_f1
    ),
    elapsed_time=elapsed_time,
)

sequential_history_df

Step 1: added feature 466, Macro F1=0.1180
Step 2: added feature 216, Macro F1=0.1556
Step 3: added feature 274, Macro F1=0.1644
Step 4: added feature 181, Macro F1=0.1662
Step 5: added feature 102, Macro F1=0.1708
Step 6: added feature 211, Macro F1=0.1722
Step 7: added feature 485, Macro F1=0.1766
Step 8: added feature 232, Macro F1=0.1785
Step 9: added feature 299, Macro F1=0.1773
Step 10: added feature 210, Macro F1=0.1810
Step 11: added feature 482, Macro F1=0.1816
Step 12: added feature 148, Macro F1=0.1823
Step 13: added feature 16, Macro F1=0.1821
Step 14: added feature 42, Macro F1=0.1836
Step 15: added feature 15, Macro F1=0.1859
Sequential Forward: 15 features, validation Macro F1=0.1859, time=46.09s


,Step,Added Feature,Selected Features,Validation Macro F1
0,1,466,1,0.118012
1,2,216,2,0.155565
2,3,274,3,0.164393
3,4,181,4,0.166203
4,5,102,5,0.170759
5,6,211,6,0.172226
6,7,485,7,0.176577
7,8,232,8,0.178464
8,9,299,9,0.177303
9,10,210,10,0.181034


In [29]:
def exhaustive_feature_selection(
    candidate_indices,
    minimum_subset_size=3,
    maximum_subset_size=5,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    best_score = -np.inf
    best_subset = None

    evaluated_subsets = 0

    history = []

    for subset_size in range(
        minimum_subset_size,
        maximum_subset_size + 1,
    ):
        print(
            f"Testing all subsets of "
            f"size {subset_size}..."
        )

        for subset in itertools.combinations(
            candidate_indices,
            subset_size,
        ):
            score = evaluate_feature_indices(
                subset
            )

            evaluated_subsets += 1

            if score > best_score:
                best_score = score
                best_subset = subset

                history.append({
                    "Evaluated Subsets": (
                        evaluated_subsets
                    ),
                    "Subset Size": (
                        subset_size
                    ),
                    "Validation Macro F1": (
                        score
                    ),
                    "Feature Indices": (
                        list(subset)
                    ),
                })

                print(
                    f"New best: "
                    f"F1={score:.4f}, "
                    f"features={subset}"
                )

    return (
        np.asarray(
            best_subset,
            dtype=int,
        ),
        best_score,
        evaluated_subsets,
        pd.DataFrame(history),
    )

In [30]:
BRUTE_FORCE_TOP_K = 10
BRUTE_FORCE_MINIMUM_SIZE = 3
BRUTE_FORCE_MAXIMUM_SIZE = 5

brute_prefilter = SelectKBest(
    score_func=f_classif,
    k=BRUTE_FORCE_TOP_K,
)

brute_prefilter.fit(
    X_feature_train,
    y_feature_train,
)

brute_candidate_indices = (
    brute_prefilter.get_support(
        indices=True
    )
)

print(
    "Brute-force candidate indices:",
    brute_candidate_indices,
)

start_time = time.time()

(
    brute_force_indices,
    brute_force_validation_f1,
    evaluated_subsets,
    brute_force_history_df,
) = exhaustive_feature_selection(
    candidate_indices=(
        brute_candidate_indices
    ),
    minimum_subset_size=(
        BRUTE_FORCE_MINIMUM_SIZE
    ),
    maximum_subset_size=(
        BRUTE_FORCE_MAXIMUM_SIZE
    ),
)

elapsed_time = time.time() - start_time

record_selection(
    method="Brute Force",
    family="Exhaustive",
    feature_indices=(
        brute_force_indices
    ),
    validation_macro_f1=(
        brute_force_validation_f1
    ),
    elapsed_time=elapsed_time,
)

print(
    "Total subsets evaluated:",
    evaluated_subsets,
)

brute_force_history_df

Brute-force candidate indices: [ 15 102 108 210 211 216 337 399 466 482]
Testing all subsets of size 3...
New best: F1=0.1361, features=(15, 102, 108)
New best: F1=0.1375, features=(15, 102, 210)
New best: F1=0.1506, features=(15, 102, 216)
New best: F1=0.1519, features=(15, 210, 216)
New best: F1=0.1539, features=(15, 211, 337)
New best: F1=0.1542, features=(102, 210, 216)
New best: F1=0.1601, features=(102, 216, 466)
New best: F1=0.1626, features=(108, 211, 216)
New best: F1=0.1664, features=(108, 216, 337)
Testing all subsets of size 4...
New best: F1=0.1672, features=(108, 211, 216, 337)
Testing all subsets of size 5...
New best: F1=0.1674, features=(15, 102, 216, 466, 482)
New best: F1=0.1691, features=(102, 108, 211, 216, 466)
Brute Force: 5 features, validation Macro F1=0.1691, time=31.95s
Total subsets evaluated: 582


,Evaluated Subsets,Subset Size,Validation Macro F1,Feature Indices
0,1,3,0.136088,"[15, 102, 108]"
1,2,3,0.137544,"[15, 102, 210]"
2,4,3,0.150595,"[15, 102, 216]"
3,17,3,0.151851,"[15, 210, 216]"
4,23,3,0.153862,"[15, 211, 337]"
5,45,3,0.154230,"[102, 210, 216]"
6,57,3,0.160088,"[102, 216, 466]"
7,71,3,0.162562,"[108, 211, 216]"
8,76,3,0.166384,"[108, 216, 337]"
9,276,4,0.167246,"[108, 211, 216, 337]"


In [31]:
selection_results_df = pd.DataFrame(
    selection_results
)

selection_results_df = (
    selection_results_df
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

selection_results_df

,Method,Family,Selected Features,Validation Macro F1,Selection Time
0,L1-SVM C=0.1,L1-SVM,509,0.290848,568.008662
1,All Features,Baseline,512,0.289964,45.303997
2,L1-SVM C=0.03,L1-SVM,459,0.287774,140.270352
3,ANOVA K=448,ANOVA,448,0.279610,36.360022
4,ANOVA K=384,ANOVA,384,0.268006,22.053366
5,ANOVA K=320,ANOVA,320,0.264406,16.305616
6,L1-SVM C=0.01,L1-SVM,330,0.262900,105.950198
7,ANOVA K=256,ANOVA,256,0.253083,9.956862
8,ANOVA K=192,ANOVA,192,0.242386,5.645946
9,L1-SVM C=0.003,L1-SVM,138,0.233655,7.968179


In [32]:
best_family_rows = (
    selection_results_df
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .groupby(
        "Family",
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

best_family_rows

,Method,Family,Selected Features,Validation Macro F1,Selection Time
0,L1-SVM C=0.1,L1-SVM,509,0.290848,568.008662
1,All Features,Baseline,512,0.289964,45.303997
2,ANOVA K=448,ANOVA,448,0.279610,36.360022
3,Sequential Forward,Sequential,15,0.185871,46.092410
4,Brute Force,Exhaustive,5,0.169103,31.947255


In [33]:
FINAL_FEATURE_SETS = {}

for _, row in best_family_rows.iterrows():
    method_name = row["Method"]

    FINAL_FEATURE_SETS[
        method_name
    ] = selected_feature_sets[
        method_name
    ]

print(
    "Final methods:",
    list(FINAL_FEATURE_SETS.keys()),
)

Final methods: ['L1-SVM C=0.1', 'All Features', 'ANOVA K=448', 'Sequential Forward', 'Brute Force']


In [34]:
MODELS = {
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(256, 128),
            early_stopping=True,
            max_iter=300,
            random_state=42,
        ),
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(
            n_neighbors=15,
            metric="cosine",
            n_jobs=-1,
        ),
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),

    "Linear SVM": make_pipeline(
        StandardScaler(),
        LinearSVC(
            class_weight="balanced",
            dual=False,
            tol=1e-3,
            max_iter=10000,
            random_state=42,
        ),
    ),
}

In [35]:
final_results = []

for method_name, feature_indices in (
    FINAL_FEATURE_SETS.items()
):
    print("\n" + "=" * 70)

    print(
        f"{method_name}: "
        f"{len(feature_indices)} features"
    )

    X_train_selected = X_train[
        :,
        feature_indices,
    ]

    X_test_selected = X_test[
        :,
        feature_indices,
    ]

    for model_name, base_model in (
        MODELS.items()
    ):
        print(
            f"Training {model_name}..."
        )

        model = clone(base_model)

        start_time = time.time()

        model.fit(
            X_train_selected,
            y_train,
        )

        predictions = model.predict(
            X_test_selected
        )

        elapsed_time = (
            time.time() - start_time
        )

        final_results.append({
            "Embedding": DATASET_NAME,
            "Feature Method": (
                method_name
            ),
            "Selected Features": len(
                feature_indices
            ),
            "Model": model_name,
            "Accuracy": accuracy_score(
                y_test,
                predictions,
            ),
            "Balanced Accuracy": (
                balanced_accuracy_score(
                    y_test,
                    predictions,
                )
            ),
            "Macro F1": f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
            "Train/Eval Time": (
                elapsed_time
            ),
        })


L1-SVM C=0.1: 509 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

All Features: 512 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

ANOVA K=448: 448 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

Sequential Forward: 15 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

Brute Force: 5 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...


In [36]:
final_results_df = pd.DataFrame(
    final_results
)

final_results_df = (
    final_results_df
    .sort_values(
        "Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_results_df

,Embedding,Feature Method,Selected Features,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time
0,Whisper Base 0.5s,ANOVA K=448,448,MLP,0.374626,0.377542,0.377589,76.798223
1,Whisper Base 0.5s,All Features,512,MLP,0.375330,0.379603,0.375847,97.779275
2,Whisper Base 0.5s,L1-SVM C=0.1,509,MLP,0.373394,0.378411,0.372378,98.157332
3,Whisper Base 0.5s,All Features,512,Random Forest,0.298610,0.311990,0.297093,52.300689
4,Whisper Base 0.5s,L1-SVM C=0.1,509,Random Forest,0.295970,0.308330,0.294328,51.902429
5,Whisper Base 0.5s,ANOVA K=448,448,Random Forest,0.294915,0.307459,0.292617,42.536129
6,Whisper Base 0.5s,All Features,512,Linear SVM,0.298786,0.316737,0.291243,100.541370
7,Whisper Base 0.5s,L1-SVM C=0.1,509,Linear SVM,0.297554,0.315531,0.290245,132.788593
8,Whisper Base 0.5s,ANOVA K=448,448,Linear SVM,0.285237,0.302877,0.276594,71.529832
9,Whisper Base 0.5s,ANOVA K=448,448,KNN,0.268696,0.269018,0.274248,3.294550


In [37]:
best_result_per_method = (
    final_results_df
    .sort_values(
        "Macro F1",
        ascending=False,
    )
    .groupby(
        "Feature Method",
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

best_result_per_method[
    [
        "Feature Method",
        "Selected Features",
        "Model",
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "Train/Eval Time",
    ]
]

,Feature Method,Selected Features,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time
0,ANOVA K=448,448,MLP,0.374626,0.377542,0.377589,76.798223
1,All Features,512,MLP,0.375330,0.379603,0.375847,97.779275
2,L1-SVM C=0.1,509,MLP,0.373394,0.378411,0.372378,98.157332
3,Sequential Forward,15,Random Forest,0.230688,0.236759,0.230404,7.499255
4,Brute Force,5,Random Forest,0.177723,0.180276,0.177465,4.760828


In [38]:
OUTPUT_DIR = Path(
    "whisper_0_5s_feature_selection_results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

selection_results_df.to_csv(
    OUTPUT_DIR
    / "validation_feature_selection.csv",
    index=False,
)

final_results_df.to_csv(
    OUTPUT_DIR
    / "heldout_test_results.csv",
    index=False,
)

best_result_per_method.to_csv(
    OUTPUT_DIR
    / "best_result_per_method.csv",
    index=False,
)

sequential_history_df.to_csv(
    OUTPUT_DIR
    / "sequential_history.csv",
    index=False,
)

brute_force_history_df.to_csv(
    OUTPUT_DIR
    / "brute_force_history.csv",
    index=False,
)

np.savez(
    OUTPUT_DIR
    / "selected_feature_indices.npz",
    **{
        method_name
        .lower()
        .replace(" ", "_")
        .replace("=", "_")
        .replace(".", "_"): indices

        for method_name, indices
        in FINAL_FEATURE_SETS.items()
    },
)

print(
    f"Saved results to: "
    f"{OUTPUT_DIR.resolve()}"
)

Saved results to: /Users/bhavaykhatri/Desktop/Assignments/audio_data_benchmarking_mml_lab/whisper_0_5s_feature_selection_results
